# Project 95 — Local Debugging Workflow Agent
## Inspect Error Logs and Suggest Likely Fixes

**Stack:** LangGraph · Ollama · LangChain · Jupyter

## Project Overview

This notebook builds a **local debugging workflow agent** that takes error logs
and bug reports as input, then follows a structured investigation process —
hypothesize → investigate → diagnose → plan fix — using a LangGraph state
machine powered by a local LLM.

Everything runs **locally** via Ollama. No logs or code leave your machine.

### What You Will Learn

1. How to **parse error logs** into structured bug context
2. How to **generate debugging hypotheses** with ranked probabilities
3. How to build a **LangGraph state-machine workflow** for multi-step reasoning
4. How to produce **root-cause analysis** grounded in investigation findings
5. How to generate **actionable fix plans** with verification steps
6. Practical patterns for agentic debugging automation

## Prerequisites

| Requirement | Details |
|---|---|
| **Ollama** | Running locally with `qwen3:8b` pulled |
| **Python packages** | `langchain`, `langchain-ollama`, `langgraph` |

```bash
ollama pull qwen3:8b
```

In [ ]:
# !pip install -q langchain langchain-ollama langgraph

## Step 1 — Imports and Configuration

In [ ]:
import json
import re
import textwrap
from typing import TypedDict, Annotated
import operator

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END

OLLAMA_MODEL = "qwen3:8b"
TEMPERATURE = 0.1

print("Configuration ready.")

## Step 2 — Initialize LLM

We use **Qwen3 8B** via Ollama with a low temperature for precise,
deterministic debugging analysis.

In [ ]:
llm = ChatOllama(model=OLLAMA_MODEL, temperature=TEMPERATURE)

test_msg = llm.invoke("Reply with only: OK")
print(f"LLM ready: {test_msg.content.strip()[:20]}")

## Step 3 — Define Sample Bug Reports

We define three realistic production bug reports, each with different
characteristics:

| Bug | Type | Complexity |
|---|---|---|
| BUG-001 | Connection pool exhaustion | Infrastructure |
| BUG-002 | Stale cache data | Configuration |
| BUG-003 | Memory leak in worker | Resource management |

In [ ]:
BUG_REPORTS = [
    {
        "id": "BUG-001",
        "title": "Users get 500 error on login",
        "description": (
            "Since deploy v2.3.1, random users get HTTP 500 on login. "
            "Happens ~10% of requests. Error log shows: "
            "'ConnectionPool: max retries exceeded with url: /auth/token'. "
            "Started after we increased session timeout from 30min to 2hrs."
        ),
        "error_log": textwrap.dedent("""\
            2026-04-14 08:23:15 ERROR [auth-service] requests.exceptions.ConnectionError:
              HTTPSConnectionPool(host='auth-svc', port=443): Max retries exceeded
              with url: /auth/token (Caused by NewConnectionError(
              '<urllib3.connection.HTTPSConnection object>: Failed to establish connection'))
            2026-04-14 08:23:15 ERROR [auth-service] Traceback (most recent call last):
              File "/app/auth/handler.py", line 42, in login
                token = token_client.create_session(user_id, ttl=7200)
              File "/app/auth/client.py", line 88, in create_session
                resp = self.session.post(self.token_url, json=payload, timeout=5)
            """),
        "env": "production, k8s, 3 replicas, auth-svc behind internal LB",
    },
    {
        "id": "BUG-002",
        "title": "Dashboard shows yesterday's numbers",
        "description": (
            "The analytics dashboard consistently shows data from the previous day. "
            "Cache was recently moved from Redis to memcached. TTL is set to 3600s. "
            "Data pipeline runs at 02:00 UTC daily. Dashboard queries hit the cache first."
        ),
        "error_log": textwrap.dedent("""\
            2026-04-14 09:15:00 INFO  [dashboard-api] Cache HIT for key=analytics:daily:2026-04-13
            2026-04-14 09:15:00 INFO  [dashboard-api] Serving cached response, age=25200s
            2026-04-14 09:15:01 WARN  [dashboard-api] Cache key analytics:daily:2026-04-14 not found
            2026-04-14 09:15:01 INFO  [dashboard-api] Fallback: serving stale key analytics:daily:2026-04-13
            """),
        "env": "production, AWS, memcached 3-node cluster",
    },
    {
        "id": "BUG-003",
        "title": "Memory leak in background job processor",
        "description": (
            "Worker pods OOM crash every ~6 hours. Memory usage grows linearly. "
            "Processing ~1000 jobs/hour. Each job downloads a file (1-50MB), processes it, "
            "and uploads the result. Using requests library for downloads."
        ),
        "error_log": textwrap.dedent("""\
            2026-04-14 14:02:33 ERROR [worker-3] OOMKilled: container exceeded 4Gi memory limit
            2026-04-14 14:02:33 INFO  [worker-3] Last 5min stats: processed=83, mem=3.8Gi, open_fds=1247
            2026-04-14 14:02:33 WARN  [worker-3] /tmp/job_files/ contains 4312 files (3.2GB total)
            2026-04-14 13:55:00 INFO  [worker-3] mem=3.5Gi, open_fds=1189
            2026-04-14 13:50:00 INFO  [worker-3] mem=3.2Gi, open_fds=1134
            """),
        "env": "production, k8s, 4Gi memory limit per pod",
    },
]

print(f"Bug reports: {len(BUG_REPORTS)}")
for bug in BUG_REPORTS:
    print(f"  {bug['id']}: {bug['title']}")

## Step 4 — Error Log Parser

Before sending to the LLM, we extract key signals from the raw error
logs: error level counts, exception types, and notable patterns.

In [ ]:
def parse_error_log(log_text: str) -> dict:
    """Extract structured signals from raw error log text."""
    lines = [l.strip() for l in log_text.strip().splitlines() if l.strip()]

    # Count log levels
    levels = {"ERROR": 0, "WARN": 0, "INFO": 0}
    for line in lines:
        for level in levels:
            if level in line:
                levels[level] += 1

    # Extract exception types
    exceptions = re.findall(r'(\w+(?:Error|Exception|Killed))', log_text)
    unique_exceptions = list(set(exceptions))

    # Extract numeric values (memory, counts, file descriptors)
    numbers = re.findall(r'(\w+)=(\d+\.?\d*\w*)', log_text)

    return {
        "total_lines": len(lines),
        "levels": levels,
        "exceptions": unique_exceptions,
        "key_values": dict(numbers[:10]),
    }


for bug in BUG_REPORTS:
    parsed = parse_error_log(bug["error_log"])
    bug["parsed_log"] = parsed
    print(f"\n{bug['id']} log summary:")
    print(f"  Lines: {parsed['total_lines']}, Levels: {parsed['levels']}")
    print(f"  Exceptions: {parsed['exceptions']}")
    print(f"  Key values: {parsed['key_values']}")

## Step 5 — Build Hypothesis Generation Chain

The first step of debugging is to **generate hypotheses** about what
might be causing the bug. We ask the LLM to produce ranked hypotheses
with supporting/opposing evidence and investigation steps.

In [ ]:
HYPOTHESIS_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a senior site reliability engineer. Given a bug report
and error logs, generate 3 ranked hypotheses for the root cause.

For each hypothesis provide:
  HYPOTHESIS: <description>
  PROBABILITY: <low|medium|high>
  EVIDENCE FOR: <what supports this>
  EVIDENCE AGAINST: <what argues against this>
  INVESTIGATE: <specific commands or checks to confirm/deny>

Also state:
  SEVERITY: P0 / P1 / P2 / P3
  CATEGORY: infrastructure / logic / data / config / resource
  IMMEDIATE MITIGATION: <what to do right now to reduce impact>

Be specific — reference error messages, metrics, and timestamps."""),
    ("human", """Bug Report:
ID: {bug_id}
Title: {title}
Description: {description}
Error Log:
{error_log}
Environment: {env}
Parsed Signals: {parsed}""")
])

hypothesis_chain = HYPOTHESIS_PROMPT | llm | StrOutputParser()

print("Hypothesis generation chain ready.")

In [ ]:
hypotheses = {}
for bug in BUG_REPORTS:
    hyp = hypothesis_chain.invoke({
        "bug_id": bug["id"],
        "title": bug["title"],
        "description": bug["description"],
        "error_log": bug["error_log"],
        "env": bug["env"],
        "parsed": json.dumps(bug["parsed_log"]),
    })
    hypotheses[bug["id"]] = hyp
    print(f"\nHYPOTHESES — {bug['id']}: {bug['title']}")
    print("=" * 60)
    print(hyp[:800])
    if len(hyp) > 800:
        print(f"... ({len(hyp)} chars total)")
    print()

## Step 6 — Build the LangGraph Debug Workflow

We build a **state-machine workflow** with three nodes:

```
investigate → diagnose → plan_fix → END
```

Each node takes the current state, calls the LLM, and returns updated state.
The state carries the bug context, investigation findings, root cause,
and fix plan through the pipeline.

In [ ]:
class DebugState(TypedDict):
    """State passed through the debugging workflow."""
    bug_id: str
    bug_title: str
    bug_description: str
    error_log: str
    env: str
    hypotheses_text: str
    investigation: str
    root_cause: str
    fix_plan: str
    status: str


# ── Node 1: Investigate ──────────────────────────────────────
INVESTIGATE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are investigating a production bug. Based on the hypotheses,
simulate running the investigation steps and report findings.

For each investigation step:
1. State what you checked
2. State what you found
3. State whether it supports or refutes each hypothesis

Be specific — include simulated command outputs, metric values, and log excerpts."""),
    ("human", "Bug: {bug_id} — {bug_title}\nDescription: {bug_description}\n"
     "Error log:\n{error_log}\nHypotheses:\n{hypotheses_text}")
])

investigate_chain = INVESTIGATE_PROMPT | llm | StrOutputParser()


def investigate(state: DebugState) -> dict:
    findings = investigate_chain.invoke({
        "bug_id": state["bug_id"],
        "bug_title": state["bug_title"],
        "bug_description": state["bug_description"],
        "error_log": state["error_log"],
        "hypotheses_text": state["hypotheses_text"],
    })
    return {"investigation": findings, "status": "investigated"}


# ── Node 2: Diagnose ─────────────────────────────────────────
DIAGNOSE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Based on the investigation findings, determine the ROOT CAUSE
of the bug. Be specific and conclusive.

State:
ROOT CAUSE: <specific technical root cause>
CONFIDENCE: <high|medium|low>
EXPLANATION: <2-3 sentences explaining the causal chain>"""),
    ("human", "Bug: {bug_id} — {bug_title}\n{bug_description}\n\n"
     "Investigation findings:\n{investigation}")
])

diagnose_chain = DIAGNOSE_PROMPT | llm | StrOutputParser()


def diagnose(state: DebugState) -> dict:
    root_cause = diagnose_chain.invoke({
        "bug_id": state["bug_id"],
        "bug_title": state["bug_title"],
        "bug_description": state["bug_description"],
        "investigation": state["investigation"],
    })
    return {"root_cause": root_cause, "status": "diagnosed"}


# ── Node 3: Plan Fix ─────────────────────────────────────────
FIX_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Create a detailed fix plan for the diagnosed root cause.

Include:
1. IMMEDIATE FIX: What to do right now (e.g., config change, restart, rollback)
2. PERMANENT FIX: Code or architecture changes needed
3. VERIFICATION: How to confirm the fix works
4. PREVENTION: How to prevent this class of bug in the future

Be specific — include example config values, code patterns, and monitoring queries."""),
    ("human", "Bug: {bug_id}\nRoot cause:\n{root_cause}\nEnvironment: {env}")
])

fix_chain = FIX_PROMPT | llm | StrOutputParser()


def plan_fix(state: DebugState) -> dict:
    fix = fix_chain.invoke({
        "bug_id": state["bug_id"],
        "root_cause": state["root_cause"],
        "env": state["env"],
    })
    return {"fix_plan": fix, "status": "fix_ready"}


# ── Build the graph ──────────────────────────────────────────
graph = StateGraph(DebugState)
graph.add_node("investigate", investigate)
graph.add_node("diagnose", diagnose)
graph.add_node("plan_fix", plan_fix)
graph.set_entry_point("investigate")
graph.add_edge("investigate", "diagnose")
graph.add_edge("diagnose", "plan_fix")
graph.add_edge("plan_fix", END)

workflow = graph.compile()

print("Debug workflow compiled: investigate → diagnose → plan_fix → END")

## Step 7 — Run Debug Workflow on BUG-001

Connection pool exhaustion after increasing session timeout.
The workflow should identify that longer sessions hold connections
open longer, exhausting the pool.

In [ ]:
def run_debug_workflow(bug: dict, hyp_text: str) -> dict:
    """Run the full debug workflow for a bug report."""
    initial_state = {
        "bug_id": bug["id"],
        "bug_title": bug["title"],
        "bug_description": bug["description"],
        "error_log": bug["error_log"],
        "env": bug["env"],
        "hypotheses_text": hyp_text,
        "investigation": "",
        "root_cause": "",
        "fix_plan": "",
        "status": "started",
    }
    result = workflow.invoke(initial_state)
    return result


bug = BUG_REPORTS[0]
result_001 = run_debug_workflow(bug, hypotheses[bug["id"]])

print(f"DEBUG RESULT — {bug['id']}: {bug['title']}")
print("=" * 60)
print(f"Status: {result_001['status']}")
print(f"\n--- Root Cause ---")
print(result_001["root_cause"][:500])
print(f"\n--- Fix Plan ---")
print(result_001["fix_plan"][:500])

## Step 8 — Run Debug Workflow on BUG-002

Stale cache after migration from Redis to memcached.
The workflow should identify cache key mismatch or TTL issues.

In [ ]:
bug = BUG_REPORTS[1]
result_002 = run_debug_workflow(bug, hypotheses[bug["id"]])

print(f"DEBUG RESULT — {bug['id']}: {bug['title']}")
print("=" * 60)
print(f"Status: {result_002['status']}")
print(f"\n--- Root Cause ---")
print(result_002["root_cause"][:500])
print(f"\n--- Fix Plan ---")
print(result_002["fix_plan"][:500])

## Step 9 — Run Debug Workflow on BUG-003

Memory leak from uncleaned temp files and possibly unclosed file handles.
The workflow should identify the growing `/tmp` directory and open FDs.

In [ ]:
bug = BUG_REPORTS[2]
result_003 = run_debug_workflow(bug, hypotheses[bug["id"]])

print(f"DEBUG RESULT — {bug['id']}: {bug['title']}")
print("=" * 60)
print(f"Status: {result_003['status']}")
print(f"\n--- Root Cause ---")
print(result_003["root_cause"][:500])
print(f"\n--- Fix Plan ---")
print(result_003["fix_plan"][:500])

## Step 10 — Debug Dashboard

We compile all debug results into a summary dashboard.

In [ ]:
all_results = [
    (BUG_REPORTS[0], result_001),
    (BUG_REPORTS[1], result_002),
    (BUG_REPORTS[2], result_003),
]

print("DEBUG WORKFLOW DASHBOARD")
print("=" * 70)
print(f"{'Bug':<10} {'Title':<40} {'Status':<12}")
print("-" * 70)

for bug, result in all_results:
    print(f"{bug['id']:<10} {bug['title'][:38]:<40} {result['status']:<12}")

print("-" * 70)
print(f"\nAll {len(all_results)} bugs processed through investigate → diagnose → plan_fix")

# Log parsing stats
print("\nERROR LOG PARSING SUMMARY")
print("-" * 50)
for bug in BUG_REPORTS:
    p = bug["parsed_log"]
    print(f"  {bug['id']}: {p['total_lines']} lines, "
          f"errors={p['levels']['ERROR']}, "
          f"exceptions={p['exceptions']}")

## Step 11 — Quality Check: Did the Agent Catch Key Issues?

We check whether the debugging agent identified the **expected root causes**
for each bug. This is a qualitative evaluation.

In [ ]:
EXPECTED_SIGNALS = [
    {
        "bug_id": "BUG-001",
        "expected_keywords": ["connection pool", "session", "timeout", "exhausted"],
        "description": "Longer sessions hold connections, exhausting the pool",
    },
    {
        "bug_id": "BUG-002",
        "expected_keywords": ["cache", "stale", "key", "TTL"],
        "description": "Cache serves stale data because new key not populated or TTL too long",
    },
    {
        "bug_id": "BUG-003",
        "expected_keywords": ["temp", "file", "cleanup", "leak"],
        "description": "Temp files not cleaned up after processing, filling disk/memory",
    },
]

print("QUALITY CHECK — Did the agent catch the expected root causes?")
print("=" * 60)

for expected, (bug, result) in zip(EXPECTED_SIGNALS, all_results):
    combined = (result["root_cause"] + " " + result["fix_plan"]).lower()
    hits = [kw for kw in expected["expected_keywords"] if kw in combined]
    coverage = len(hits) / len(expected["expected_keywords"])

    status = "PASS" if coverage >= 0.5 else "PARTIAL" if coverage > 0 else "MISS"
    print(f"\n{expected['bug_id']}: {status} ({coverage:.0%} keyword coverage)")
    print(f"  Expected: {expected['description']}")
    print(f"  Keywords found: {hits}")
    print(f"  Keywords missed: {[k for k in expected['expected_keywords'] if k not in combined]}")

## Evaluation Summary

| Dimension | How We Evaluated |
|---|---|
| **Log parsing** | Verified `parse_error_log()` extracts correct exception types and metrics |
| **Hypothesis quality** | Checked hypotheses reference specific error messages and config changes |
| **Workflow execution** | All 3 bugs processed through the full investigate → diagnose → plan_fix pipeline |
| **Root cause accuracy** | Keyword-based check against expected root causes (Step 11) |
| **Fix plan quality** | Inspected whether plans include immediate + permanent fixes and verification |
| **LangGraph structure** | Verified state flows correctly through all 3 nodes |

### Known Limitations

- **Simulated investigation**: The agent cannot actually run commands or
  query real infrastructure — it simulates what investigation would find.
- **No real logs**: With access to real logging systems (ELK, Datadog),
  the agent could retrieve actual metrics and traces.
- **Single pass**: The workflow runs linearly. A more advanced version
  could loop back to investigate if the diagnosis confidence is low.
- **No code context**: The agent sees error logs but not the full source
  code, limiting its ability to suggest precise code fixes.
- **Hallucination risk**: The agent may confidently diagnose incorrect
  root causes, especially for ambiguous bugs.

## How to Improve This Project

1. **Real log integration** — connect to ELK/Loki/CloudWatch to fetch
   actual logs and metrics during the investigate step
2. **Conditional loops** — add a LangGraph conditional edge that re-investigates
   if the diagnosis confidence is low
3. **Tool calling** — give the agent tools (kubectl, SQL queries, API calls)
   to actually investigate infrastructure state
4. **Multi-agent** — use separate agents for different specialties
   (infrastructure, application logic, data pipeline)
5. **Knowledge base** — connect to a runbook database so the agent can
   match known patterns to historical fixes
6. **Slack/PagerDuty integration** — trigger the workflow from alerts
   and post results back to incident channels

## What You Learned

- **Error log parsing** — extracting structured signals from raw log text
- **Hypothesis-driven debugging** — generating ranked theories with evidence
- **LangGraph state machines** — building multi-step agentic workflows
- **Root cause analysis** — diagnosing bugs from simulated investigation findings
- **Fix planning** — generating actionable fixes with verification steps
- **Quality evaluation** — checking agent output against expected root causes